# SPATA2-style spatial utilities in OmicVerse

This notebook demonstrates the AnnData-native `ov.space.spata2_*` utilities: coordinate extraction, variable joins, tissue outlines, spatial outlier filtering, and explicit pixel/unit conversion.

The API is inspired by SPATA2's object and spatial-data foundation, but it runs fully in Python and does not require R or `rpy2`.


In [ ]:
import numpy as np
import pandas as pd
from anndata import AnnData
import omicverse as ov


## Create a small spatial AnnData object

Real workflows usually start from `scanpy.read_visium`, OmicVerse I/O helpers, Xenium, MERFISH, or other spatial loaders. Here we use a tiny fixture so the example is self-contained.


In [ ]:
coords = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
    [0.5, 0.5],
    [12.0, 12.0],  # isolated artifact-like spot
])
counts = np.array([
    [1.0, 0.0, 5.0],
    [2.0, 1.0, 4.0],
    [3.0, 1.0, 3.0],
    [4.0, 2.0, 2.0],
    [5.0, 3.0, 1.0],
    [8.0, 0.0, 0.0],
])
obs = pd.DataFrame(
    {
        "region": ["core", "core", "edge", "edge", "core", "artifact"],
        "total_counts": counts.sum(axis=1),
    },
    index=[f"spot_{idx}" for idx in range(coords.shape[0])],
)
adata = AnnData(X=counts, obs=obs, var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]))
adata.obsm["spatial"] = coords
adata


## Coordinates and molecular variables


In [ ]:
ov.space.spata2_get_coords(adata, include_obs=["region"])


In [ ]:
ov.space.spata2_join_variables(adata, ["region", "GeneA", "GeneB"])


## Tissue outline and spatial outliers


In [ ]:
outline = ov.space.spata2_tissue_outline(adata)
outline


In [ ]:
outliers = ov.space.spata2_identify_outliers(adata, radius=1.6, min_neighbors=2)
outliers


In [ ]:
filtered = ov.space.spata2_remove_outliers(adata)
filtered.obs_names.tolist()


## Explicit distance conversion


In [ ]:
pixels = np.array([0.0, 10.0, 25.0])
units = ov.space.spata2_pixels_to_unit(pixels, pixels_per_unit=5.0)
ov.space.spata2_unit_to_pixels(units, pixels_per_unit=5.0)
